<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W6D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# Daily Challenge: Fine-Tuning LLM with LoRA (One Cell)
# ==========================================================

# Install required libraries
%pip install -q peft==0.4.0 datasets transformers accelerate

# Create cache directory
import os
os.makedirs("cache", exist_ok=True)

# ==========================================================
# Import libraries
# ==========================================================

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)

import transformers
import time
import torch

# ==========================================================
# Load model and tokenizer
# ==========================================================

model_name = "bigscience/bloomz-560m"

tokenizer = AutoTokenizer.from_pretrained(model_name)

foundation_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

# ==========================================================
# Load dataset (10% sample)
# ==========================================================

data = load_dataset("Abirate/english_quotes", split="train")

sample_size = int(len(data) * 0.10)
data = data.select(range(sample_size))

# Tokenize dataset
data = data.map(
    lambda samples: tokenizer(
        samples["quote"],
        truncation=True,
        padding="max_length",
        max_length=64
    ),
    batched=True
)

# Small training sample (as requested)
train_sample = data.select(range(5))

display(train_sample)

# ==========================================================
# Configure LoRA
# ==========================================================

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# ==========================================================
# Apply LoRA
# ==========================================================

peft_model = get_peft_model(
    foundation_model,
    lora_config
)

peft_model.print_trainable_parameters()

# ==========================================================
# Training Arguments
# ==========================================================

output_directory = os.path.join(
    "cache",
    "peft_lab_outputs"
)

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=3,
    use_cpu=True
)

# ==========================================================
# Trainer
# ==========================================================

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer,
        mlm=False
    )
)

# ==========================================================
# Train model
# ==========================================================

trainer.train()

# ==========================================================
# Save LoRA model
# ==========================================================

time_now = int(time.time())

peft_model_path = os.path.join(
    output_directory,
    f"peft_model_{time_now}"
)

trainer.model.save_pretrained(
    peft_model_path
)

# ==========================================================
# Load saved LoRA model
# ==========================================================

loaded_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

loaded_peft_model = PeftModel.from_pretrained(
    loaded_model,
    peft_model_path,
    is_trainable=False
)

# ==========================================================
# Inference
# ==========================================================

inputs = tokenizer(
    "Two things are infinite: ",
    return_tensors="pt"
)

outputs = loaded_peft_model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7
)

print(
    tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )
)